In [0]:
# catalog -> volume -> file -> table

catalog = "ecommerce_dev"

volume = f"/Volumes/{catalog}/bronze/landing_zone"

customers_file = f"{volume}/customers.csv"
orders_file = f"{volume}/orders.csv"
products_file = f"{volume}/products.csv"

In [0]:
from pyspark.sql.functions import col, current_timestamp        

In [0]:
df_orders_raw = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","false")
    .load(orders_file)
)

In [0]:
df_customers_raw = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","false")
    .load(customers_file)
)

In [0]:
df_products_raw = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","false")
    .load(products_file)
)

In [0]:
df_customers_raw.printSchema()
df_orders_raw.printSchema()
df_products_raw.printSchema()

### Insert Timestamp col and Source File col

In [0]:
df_orders_bronze = (
    df_orders_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
df_customers_bronze = (
    df_customers_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
df_products_bronze = (
    df_products_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
display(df_orders_bronze)

In [0]:
display(df_customers_bronze)

In [0]:
display(df_products_bronze)

### Take the Bronze DataFrame → convert it into Delta format → append the data → save it as a table in Databricks.

In [0]:
bronze_orders_table = f"{catalog}.bronze.orders"

(
    df_orders_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_orders_table)
)

In [0]:
bronze_customers_table = f"{catalog}.bronze.customers"

(
    df_customers_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_customers_table)
)

In [0]:
bronze_products_table = f"{catalog}.bronze.products"

(
    df_products_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_products_table)
)

In [0]:
print("Loaded Successfully:")
print(bronze_orders_table)
print(bronze_customers_table)
print(bronze_products_table)

In [0]:
print("\nRow Counts:")
print(f"{bronze_orders_table}: {spark.table(bronze_orders_table).count()}")
print(f"{bronze_customers_table}: {spark.table(bronze_customers_table).count()}")
print(f"{bronze_products_table}: {spark.table(bronze_products_table).count()}")

In [0]:
print("Orders Schema:")
spark.table(bronze_orders_table).printSchema()

In [0]:
print("Customers Schema:")
spark.table(bronze_customers_table).printSchema()

In [0]:
print("Products Schema:")
spark.table(bronze_products_table).printSchema()

In [0]:
display(
    spark.table(bronze_orders_table)
    .select("_ingestion_timestamp","_source_file")
    .distinct()
)

In [0]:
display(
    spark.table(bronze_customers_table)
    .select("_ingestion_timestamp","_source_file")
    .distinct()
)

In [0]:
display(
    spark.table(bronze_products_table)
    .select("_ingestion_timestamp","_source_file")
    .distinct()
)